# Project: steer the qubit
**Quantum Computing · Week 2 (Gates and the Bloch sphere) · Day 5**

Lesson page: https://sara-aminpour.com/tutorials/qc/w2d5.html

In [ ]:
# On Google Colab, this installs qiskit (about a minute). On your own computer, skip it and
# install the packages once instead: pip install qiskit==2.5.2 scikit-learn torch matplotlib
import sys
if "google.colab" in sys.modules:
    %pip install --quiet qiskit==2.5.2

Project: steer the qubit. Every one-qubit state can be made from $|0\rangle$ with two rotations:
$R_Y(\theta)$ sets the latitude, then $R_Z(\varphi)$ sets the longitude.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, state_fidelity

def prepare(theta, phi):
    """|0> -> RY(theta) -> RZ(phi): the point at latitude theta, longitude phi."""
    qc = QuantumCircuit(1)
    qc.ry(theta, 0)
    qc.rz(phi, 0)
    return qc

theta, phi = np.radians(120), np.radians(45)
made = Statevector(prepare(theta, phi))
target = Statevector([np.cos(theta / 2), np.exp(1j * phi) * np.sin(theta / 2)])
print(prepare(theta, phi).draw("text"))
print("prepared:", np.round(made.data, 3))
print("target:  ", np.round(target.data, 3))
print("the same state, up to a global phase?", made.equiv(target))

The amplitudes look different, yet `equiv` says they're the same state: $R_Z$ adds a **global** phase
$e^{-i\varphi/2}$ to both amplitudes, which no experiment can see.

## How close is close?
The **fidelity** $|\langle\text{target}|\psi\rangle|^2$ is 1 for the same state and 0 for opposite points.

In [ ]:
for err in [0, 5, 20, 45, 90]:
    near = Statevector(prepare(theta + np.radians(err), phi))
    print(f"tilt off by {err:2d} degrees: fidelity {state_fidelity(near, target):.3f}")

## Project: a state compiler
Write `angles_for(state)`: given any one-qubit state, return the $(\theta, \varphi)$ that `prepare`
needs. Read them off the amplitudes $\alpha = \cos\frac\theta2$ and $\beta = e^{i\varphi}\sin\frac\theta2$
(after removing the global phase of $\alpha$). Then test it on five random states.

In [ ]:
def angles_for(state):
    a, b = state.data
    theta = 0.0   # TODO: 2 * arccos(|a|)   (np.arccos, abs)
    phi = 0.0     # TODO: the phase of b minus the phase of a   (np.angle)
    return theta, phi

rng = np.random.default_rng(7)
for _ in range(5):
    v = rng.normal(size=2) + 1j * rng.normal(size=2)
    goal = Statevector(v / np.linalg.norm(v))          # a random state
    th, ph = angles_for(goal)
    print(f"theta = {np.degrees(th):6.1f}, phi = {np.degrees(ph):7.1f}  ->  fidelity {state_fidelity(Statevector(prepare(th, ph)), goal):.4f}")

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
def angles_for(state):
    a, b = state.data
    theta = 2 * np.arccos(np.clip(abs(a), 0, 1))
    phi = np.angle(b) - np.angle(a)
    return theta, phi

rng = np.random.default_rng(7)
for _ in range(5):
    v = rng.normal(size=2) + 1j * rng.normal(size=2)
    goal = Statevector(v / np.linalg.norm(v))
    th, ph = angles_for(goal)
    print(f"theta = {np.degrees(th):6.1f}, phi = {np.degrees(ph):7.1f}  ->  fidelity {state_fidelity(Statevector(prepare(th, ph)), goal):.4f}")
print("Every random state is reached with fidelity 1: two rotations are enough for any one-qubit state.")